# Sagemaker Training and Deployment

This notebook takes the winning CNN from the main notebook (the one with pooling, 51.87% test accuracy) and trains and deploys it using Sagemaker, running from my AWS Academy Learner Lab account.

I'm running this from my local machine, not inside Sagemaker Studio, using temporary credentials from the Learner Lab saved in `~/.aws/credentials`. The actual training and hosting still happen on AWS, only the orchestration code runs locally.

## Setup

I need the account ID, region, and the `LabRole` ARN, since Learner Lab doesn't let me create a new IAM role or use `sagemaker.get_execution_role()` the way Sagemaker Studio does automatically.

In [2]:
import sagemaker
from sagemaker.tensorflow import TensorFlow

session = sagemaker.Session()
region = session.boto_region_name
account_id = session.account_id()
role = f"arn:aws:iam::{account_id}:role/LabRole"

print("Region:", region)
print("Account:", account_id)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\maria\AppData\Local\sagemaker\sagemaker\config.yaml


c:\Users\maria\AppData\Local\Programs\Python\Python311\Lib\site-packages\sagemaker\__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Region: us-east-1
Account: 957594481990
Role: arn:aws:iam::957594481990:role/LabRole


## Upload the preprocessed data to S3

The `.npy` files come from the main notebook's section 6.1. `sagemaker.Session().default_bucket()` creates (or reuses) a bucket named `sagemaker-<region>-<account-id>` automatically, so I don't have to set one up by hand.

In [3]:
bucket = session.default_bucket()
prefix = "fer2013-cnn"

train_s3_path = session.upload_data(
    path="../sagemaker_data/X_train.npy",
    bucket=bucket,
    key_prefix=f"{prefix}/train",
)
session.upload_data(
    path="../sagemaker_data/y_train.npy",
    bucket=bucket,
    key_prefix=f"{prefix}/train",
)
test_s3_path = session.upload_data(
    path="../sagemaker_data/X_test.npy",
    bucket=bucket,
    key_prefix=f"{prefix}/test",
)
session.upload_data(
    path="../sagemaker_data/y_test.npy",
    bucket=bucket,
    key_prefix=f"{prefix}/test",
)

# upload_data returns the S3 path of the last file uploaded to that prefix,
# but train_s3_path/test_s3_path point at the *folder*, which is what the
# estimator needs (it downloads everything under that prefix into the
# container, both the X and y files land in the same local folder there).
train_s3_folder = f"s3://{bucket}/{prefix}/train"
test_s3_folder = f"s3://{bucket}/{prefix}/test"
print("Train data:", train_s3_folder)
print("Test data:", test_s3_folder)

Train data: s3://sagemaker-us-east-1-957594481990/fer2013-cnn/train
Test data: s3://sagemaker-us-east-1-957594481990/fer2013-cnn/test


## Launch the training job

`TensorFlow` here is Sagemaker's prebuilt, managed TensorFlow container, I only give it my `train.py` script. `instance_type="ml.m5.large"` is a small CPU instance (Learner Lab doesn't allow GPU instances, and this CNN doesn't need one). `.fit()` is what actually launches the job, this is the part that bills by the minute while it runs.

In [5]:
estimator = TensorFlow(
    entry_point="train.py",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="2.13",
    py_version="py310",
    hyperparameters={"epochs": 15, "batch-size": 64},
)

estimator.fit({"train": train_s3_folder, "test": test_s3_folder})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: tensorflow-training-2026-08-17-04-27-17-346


2026-08-17 04:27:34 Starting - Starting the training job...
2026-08-17 04:27:48 Starting - Preparing the instances for training...
2026-08-17 04:28:11 Downloading - Downloading input data...
2026-08-17 04:29:11 Downloading - Downloading the training image........2026-08-17 04:30:29.154057: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-17 04:30:32,438 sagemaker-training-toolkit INFO     Imported framework sagemaker_tensorflow_container.training
2026-08-17 04:30:32,439 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-08-17 04:30:32,440 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-08-17 04:30:32,843 sagemaker-training-toolkit INFO     No GPUs detected (normal if

## Deploy the trained model to an endpoint

`.deploy()` takes the model this training job produced and hosts it behind a real-time inference endpoint. This is the part that bills **by the hour, continuously, until I delete it**, not just while I'm using it.

In [6]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)
print("Endpoint name:", predictor.endpoint_name)

c:\Users\maria\AppData\Local\Programs\Python\Python311\Lib\site-packages\sagemaker\model.py:347: SageMakerV2DeprecationWarning: TensorFlowModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.tensorflow.model:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating model with name: tensorflow-training-2026-08-17-04-43-23-715
INFO:sagemaker:Creating endpoint-config with name tensorflow-training-2026-08-17-04-43-23-715
INFO:sagemaker:Creating endpoint with name tensorflow-training-2026-08-

-----!

c:\Users\maria\AppData\Local\Programs\Python\Python311\Lib\site-packages\sagemaker\base_predictor.py:140: SageMakerV2DeprecationWarning: TensorFlowPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint name: tensorflow-training-2026-08-17-04-43-23-715


## Test the endpoint

I send one preprocessed test image (same normalization as the rest of the notebook: divided by 255, shaped to `(1, 48, 48, 1)`) and check that it returns 7 class probabilities.

In [7]:
import numpy as np

X_test_sample = np.load("../sagemaker_data/X_test.npy")[:1]

result = predictor.predict(X_test_sample)
print(result)

{'predictions': [[0.000833610306, 2.51972471e-10, 0.00103786192, 0.580046654, 0.000669040717, 0.417405039, 7.75950866e-06]]}


## Clean up

**This step matters.** The endpoint keeps billing every hour it stays up, whether or not I'm using it. I delete it as soon as I'm done testing.

In [8]:
predictor.delete_endpoint()
print("Endpoint deleted.")

INFO:sagemaker:Deleting endpoint configuration with name: tensorflow-training-2026-08-17-04-43-23-715
INFO:sagemaker:Deleting endpoint with name: tensorflow-training-2026-08-17-04-43-23-715


Endpoint deleted.
